In [37]:
import numpy as np
import pandas as pd
import json
import sys
from scipy.special import softmax
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

with open('data/mapping.json', 'r') as json_file:
    data_dict = json.load(json_file)

def ohe(df, target : list):
    
    for key in data_dict.keys():
        print(key)
        if (key in target):
            continue
        possible_values = data_dict[key]
        one_hot = pd.get_dummies(df[key])
        one_hot = one_hot.reindex(columns=sorted(possible_values),fill_value=False)
        one_hot = one_hot.iloc[:,1:]
        one_hot = one_hot.astype(int)
        one_hot.columns = [f"{key}_{col}" for col in one_hot.columns]
        one_hot_array = one_hot.values
        column_index = df.columns.get_loc(key)
        df = df.drop(columns=[key])
        df_array = df.values
        columns_before = df_array[:, :column_index]
        columns_after = df_array[:, column_index:]
        new_columns = list(df.columns[:column_index]) + list(one_hot.columns) + list(df.columns[column_index:])
        combined_array = np.hstack([df_array[:, :column_index], one_hot_array, df_array[:, column_index:]])
        df = pd.DataFrame(combined_array, columns=new_columns)
        print(df.shape)
    
    return df

def sigmoid(y : np.array):
    return (1/(1+np.exp(-y)))

def loss(X : np.array, w : np.array, y : np.array, lam : np.float64):
    n = y.shape[0]
    y_hat = X@w.T
    return (1/n)*(np.sum(np.log(1 + np.exp(-y*y_hat))) + lam*(w.T@w))

def gradient(X : np.array, w : np.array, y : np.array, lam : np.float64):
    y_hat = X@w.T
    return X.T@(-y*(1-sigmoid(y*y_hat))) + 2*lam*w

def mini_batch(epochs : int, batch_size : int, X : np.array, y : np.array, weights : np.array, alpha : np.float64, beta_1 : np.float64, beta_2 : np.float64, epsilon : np.float64, lam : np.float64):
    # alpha = 0.01
    # beta_1 = 0.9
    # beta_2 = 0.99
    m = 0
    v = 0
    # epsilon = 1e-8
    n = y.shape[0]
    no_of_batches = int((n+batch_size-1)/batch_size)
    for i in range(epochs):
        alpha_t = alpha*(1-np.power(beta_2, i+1))/np.power(1-np.power(beta_1, i+1), 2)
        for j in range(no_of_batches):
            X_train = X[j*batch_size : min((j+1)*batch_size, n)]
            y_train = y[j*batch_size : min((j+1)*batch_size, n)]
            grad = gradient(X_train, weights, y_train, lam)
            m = beta_1*m + (1-beta_1)*grad
            v = beta_2*v + (1-beta_2)*(grad*grad)
            
            modified_gradient = m/(np.sqrt(v) + epsilon*(np.power(1-np.power(beta_2, i+1), 0.5)))
            weights = weights - alpha_t*modified_gradient
        
        print(loss(X, weights, y, lam))
    
    return weights

            

train_file = 'data/train2.csv'                       # sys.argv[1]
#parameter = sys.argv[2]
#modelweight = sys.argv[3]
test_file =  'data/test2.csv'                                      # sys.argv[2]
df_train_origial = pd.read_csv(train_file)
df_train_origial = ohe(df_train_origial, ["Gender"])
X_train_original = df_train_origial.drop(columns= 'Gender').to_numpy(dtype=np.float64)
y_train_original = df_train_origial['Gender'].to_numpy(dtype=np.float64)
X_train_withbias = np.insert(X_train_original, 0, 1, axis=1)
scaler  = StandardScaler().fit(X_train_withbias)
X_train_withbias = scaler.transform(X_train_withbias)
unique_values, counts = np.unique(y_train_original, return_counts=True)
counts = np.array(counts, dtype = np.float64)
unique_values_size = unique_values.size
#X_train_final = X_change(X_train_withbias, y_train_original, counts, unique_values_size)


df_test_origial = pd.read_csv(test_file)
df_test_origial = ohe(df_test_origial, ["Gender"])
X_test_original = df_test_origial.to_numpy()
X_test_withbias = np.insert(X_test_original, 0, 1, axis=1)
X_test_withbias = scaler.transform(X_test_withbias)

Hospital Service Area
(87595, 32)
Hospital County
(87595, 86)
Facility Name
(87595, 285)
Age Group
(87595, 288)
Zip Code - 3 digits
(87595, 336)
Gender
Race
(87595, 338)
Ethnicity
(87595, 340)
Type of Admission
(87595, 344)
Patient Disposition
(87595, 361)
CCSR Diagnosis Code
(87595, 839)
CCSR Procedure Code
(87595, 1158)
APR Severity of Illness Description
(87595, 1160)
APR Risk of Mortality
(87595, 1162)
APR Medical Surgical Description
(87595, 1163)
Payment Typology 1
(87595, 1170)
Payment Typology 2
(87595, 1178)
Payment Typology 3
(87595, 1186)
Emergency Department Indicator
(87595, 1186)
Hospital Service Area
(58397, 31)
Hospital County
(58397, 85)
Facility Name
(58397, 284)
Age Group
(58397, 287)
Zip Code - 3 digits
(58397, 335)
Gender
Race
(58397, 337)
Ethnicity
(58397, 339)
Type of Admission
(58397, 343)
Patient Disposition
(58397, 360)
CCSR Diagnosis Code
(58397, 838)
CCSR Procedure Code
(58397, 1157)
APR Severity of Illness Description
(58397, 1159)
APR Risk of Mortality
(58

In [40]:
pca = PCA(n_components= 999)
X_train_withbias = pca.fit_transform(X_train_withbias)
X_test_withbias = pca.transform(X_test_withbias)

no_of_features = X_train_withbias.shape[1]
#y_train_ohe = one_hot_encoding(y_train_original, unique_values_size)
weights = np.zeros(no_of_features, dtype= np.float64)
n = y_train_original.shape[0]

a = 0.001
b1 = 0.99
b2 = 0.999
eps = 1e-8
lam = 0
weights = mini_batch(25,800, X_train_withbias,y_train_original, weights, a, b1, b2, eps, lam)

y_pred = sigmoid(X_test_withbias@weights)

f = open("out.txt", "w")
for i in y_pred:
    if(i >= 0.5):
        f.write('1' + '\n')
    else:
        f.write('-1\n')

f.close() 

transformation_matrix = pca.components_  # Get the transformation matrix
# original_feature_indices = np.arange(X.shape[1])  # Store original indices\
print(transformation_matrix)
# print(original_feature_indices)


0.6729128375283784
0.5998772571570564
0.5621432970803338
0.55093293400837
0.5458108086906572
0.5439405270886254
0.5431800788279472
0.5426993248520575
0.5423433916600289
0.5420751107285499
0.5418567057800039
0.541684490305871
0.5415520793006865
0.5414387483284938
0.5413378370146529
0.5412467920297902
0.5411635395965835
0.5410869267173446
0.5410160777315779
0.5409503050141132
0.5408891262464194
0.5408322328551597
0.540779427257864
0.540730526310651
0.5406852528165453
[[ 1.00000000e+00 -1.06173018e-16  8.33632635e-18 ... -3.58381433e-19
   1.25217609e-19 -8.88037527e-19]
 [ 1.06173018e-16  1.00000000e+00 -2.44249065e-15 ...  7.38786147e-18
   1.72878987e-17  3.43315692e-17]
 [-8.33632635e-18  2.09554596e-15  1.00000000e+00 ...  2.32013706e-17
   1.16293114e-16  1.42111502e-16]
 ...
 [ 3.58381433e-19 -1.64798730e-17 -1.12757026e-17 ...  1.00000000e+00
  -1.80827575e-14 -2.24820162e-15]
 [-1.25217609e-19 -1.56125113e-17 -1.20129601e-16 ...  1.79890824e-14
   1.00000000e+00  2.16771046e-14]
